In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
os.chdir("/content/drive/MyDrive/Colab Notebooks/ML_for_Architects/")

In [ ]:
# Making working directory, it will not create new folder again if the folder is already there.
os.makedirs("/content/drive/MyDrive/Colab Notebooks/ML_for_Architects/collecting data", exist_ok=True)
os.chdir("/content/drive/MyDrive/Colab Notebooks/ML_for_Architects/collecting data")

In [ ]:
import os
os.getcwd()

### Task 1: Data Collection through Request and BeautifulSoup

We collect articles from **Dezeen** (dezeen.com). Every Dezeen article has the date in its address, for example
`https://www.dezeen.com/2026/09/09/upcoming-nfl-stadiums-american-football/`, so we can ask for **all the articles of one day**.

Dezeen protects its web pages against scripts (a "Just a moment..." check by Cloudflare), so `requests` cannot read the article pages directly. But like most news sites, Dezeen also publishes its articles as an **RSS feed**: `https://www.dezeen.com/feed/?paged=1` is the newest 50 articles, `?paged=2` the 50 before, and so on. Each entry contains the full article HTML. We walk through the feed pages until we reach our date.

**Pick a date below (year/month/day). Choose one within the last two months, older dates need many feed pages.**

In [ ]:
# Pick a date after 2026/08/01

date = "2026/09/01"     # <- your date, in the form year/month/day

In [ ]:
import re, json, requests
import xml.etree.ElementTree as ET
from bs4 import BeautifulSoup

CONTENT = "{http://purl.org/rss/1.0/modules/content/}encoded"     # the feed entry that holds the article HTML


def article_from_item(item):
    """Turn one feed entry (an <item> element) into a dictionary with the text and the image addresses."""
    # BeautifulSoup parses the article HTML so we can search it
    soup = BeautifulSoup(item.findtext(CONTENT) or "", "html.parser")
    # every <p> is a paragraph; short ones are captions or credits, so we skip them
    paragraphs = [p.get_text(" ", strip=True) for p in soup.find_all("p")]
    paragraphs = [t for t in paragraphs if len(t) > 40]
    # image addresses, without the small thumbnails of the "related articles" (their address ends with -191x191.jpg)
    images = [img["src"] for img in soup.find_all("img") if img.get("src") and "-191x191" not in img["src"]]
    return {"title": item.findtext("title"), "url": item.findtext("link"),
            "published": item.findtext("pubDate"), "paragraphs": paragraphs, "images": images}


def articles_on(date, max_pages=40):
    """All Dezeen articles published on `date` (year/month/day), read from the feed pages."""
    articles, page = [], 1
    while page <= max_pages:
        response = requests.get("https://www.dezeen.com/feed/", params={"paged": page}, headers={"User-Agent": "Mozilla/5.0"})
        if response.status_code != 200:
            print(f"Error: {response.status_code} on feed page {page}")
            break
        feed = ET.fromstring(response.content.strip())                # the feed page is XML; strip() removes a blank before its header
        items = list(feed.iter("item"))                               # the entries of this feed page
        if not items:
            break
        for item in items:
            if f"/{date}/" in item.findtext("link"):                  # the date is part of the article address
                articles.append(article_from_item(item))
        # the feed is newest first: the last entry of the page is the oldest one
        oldest = re.search(r"dezeen\.com/(\d{4}/\d{2}/\d{2})/", items[-1].findtext("link")).group(1)
        print(f"feed page {page}: {len(items)} entries, oldest {oldest}, {len(articles)} articles of {date} so far")
        if oldest < date:                                               # we have passed our date: stop
            break
        page += 1
    return articles


articles = articles_on(date)
print(f"\n{len(articles)} articles published on {date}\n")

# Save every article as a text file and a JSON file, then show the titles
for x, article in enumerate(articles):
    with open(f"extracted_text_{x}.txt", "w", encoding="utf-8") as file:
        file.write(article["title"] + "\n" + article["url"] + "\n\n" + "\n".join(article["paragraphs"]) + "\n")
    with open(f"extracted_text_{x}.json", "w", encoding="utf-8") as file:
        json.dump(article, file, ensure_ascii=False, indent=2)
    print(f"[{x}] {article['title']}  ({len(article['paragraphs'])} paragraphs, {len(article['images'])} images)")
    print(f"     {article['url']}")

**Now we merge all the JSON files into one JSON file, and all the TXT files into one TXT file**

In [ ]:
json_name = "all_extracted_text.json"
txt_name = "all_extracted_text.txt"

import glob, json, re

# Find all the JSON and TXT files in the working folder (the merged files themselves are left out,
# so the cell can be run again). They are sorted by the number in the file name: _0, _1, _2, ...
def number_in(name):
    match = re.search(r"(\d+)", name)
    return int(match.group(1)) if match else -1

json_files = sorted(glob.glob("extracted_text_*.json"), key=number_in)
txt_files = sorted(glob.glob("extracted_text_*.txt"), key=number_in)
print("JSON files found:", json_files)
print("TXT files found: ", txt_files)

# Merge the JSON files into one list: one dictionary per article
merged = []
for path in json_files:
    with open(path, encoding="utf-8") as file:
        merged.append(json.load(file))
with open(json_name, "w", encoding="utf-8") as file:
    json.dump(merged, file, ensure_ascii=False, indent=2)

# Merge the TXT files into one text file, with a header line before the content of each file
with open(txt_name, "w", encoding="utf-8") as out:
    for path in txt_files:
        with open(path, encoding="utf-8") as file:
            out.write(f"===== {path} =====\n{file.read().strip()}\n\n")

print(f"Merged {len(json_files)} JSON files -> {json_name} ({sum(len(a['paragraphs']) for a in merged)} paragraphs)")
print(f"Merged {len(txt_files)} TXT files  -> {txt_name}")

**Please download the "all_extracted_text.json" and "all_extracted_text.txt" from your Google Drive, you need to upload these two files to the assignment.**

### Task 2: Data Collection through Scrapy - text

Same goal as Task 1, now with **Scrapy**, a framework made for crawling. A Scrapy *spider* starts at some URLs, reads each response with a `parse()` method, and can *follow* further links: here it reads feed page 1, then page 2, and so on, until it has passed our `date`.

A spider runs as a separate program, so it cannot see the variables of this notebook: we give it the date as an argument (`-a date=...`) when we run it. The `date` from Task 1 is reused.

In [ ]:
pip install scrapy

In [ ]:
%%writefile my_spider.py
# The line above writes this cell to the file my_spider.py instead of running it.
import re
import scrapy


class DezeenTextSpider(scrapy.Spider):
    # The spider is given a name, which is used to identify it when running.
    name = "my_spider_text"

    # The URL(s) from which the spider starts: the newest page of the Dezeen feed.
    start_urls = ["https://www.dezeen.com/feed/?paged=1"]

    custom_settings = {"USER_AGENT": "Mozilla/5.0", "DOWNLOAD_DELAY": 1}

    # The date arrives from the command line: scrapy runspider my_spider.py -a date=2026/09/09
    def __init__(self, date="2026/09/09", **kwargs):
        super().__init__(**kwargs)
        self.date = date

    # parse() is called with each feed page. It yields the articles of our date, then follows the next page if needed.
    def parse(self, response, page=1):
        response.selector.remove_namespaces()                     # so that <content:encoded> is simply "encoded"
        items = response.xpath("//item")
        for item in items:
            link = item.xpath("link/text()").get()
            if f"/{self.date}/" in link:                          # the date is part of the article address
                html = item.xpath("encoded/text()").get()
                # every <p> of the article; string(.) gives the full text of a paragraph, even with <a> or <em> inside
                paragraphs = [p.xpath("string(.)").get().strip() for p in scrapy.Selector(text=html).css("p")]
                yield {
                    "title": item.xpath("title/text()").get(),
                    "url": link,
                    "content": [t for t in paragraphs if len(t) > 40],
                }
        # the feed is newest first: if the oldest entry of this page is still after our date, read the next page
        if items:
            oldest = re.search(r"dezeen\.com/(\d{4}/\d{2}/\d{2})/", items[-1].xpath("link/text()").get()).group(1)
            if oldest >= self.date:
                yield response.follow(f"https://www.dezeen.com/feed/?paged={page + 1}", callback=self.parse, cb_kwargs={"page": page + 1})

In [ ]:
import json, sys

# Run the spider as a separate program (Scrapy cannot be restarted inside a notebook).
# -a date=... hands our date to the spider; -O writes all results to one JSON file; --loglevel=WARNING hides the routine log lines.
!"{sys.executable}" -m scrapy runspider my_spider.py -a date={date} -O all_scrapy_output.json --loglevel=WARNING

with open("all_scrapy_output.json", encoding="utf-8") as file:
    results = json.load(file)

for item in results:
    print(item["title"])
    print("  ", item["url"])
    for paragraph in item["content"][:2]:
        print("   -", paragraph[:100])
    print()
print(f"{len(results)} articles of {date} scraped -> all_scrapy_output.json")

**Please download the "all_scrapy_output.json" from your Google Drive, you need to upload this file to the assignment.**

### Task 3: Data Collection through Scrapy - images

The same spider idea, now collecting the **images** of the articles of our `date`. Scrapy's built-in `ImagesPipeline` downloads every address found in the `image_urls` field of an item and saves the files in the folder `images/full/`.

In [ ]:
%%writefile my_image_spider.py
# The line above writes this cell to the file my_image_spider.py instead of running it.
import re
import scrapy


class DezeenImagesSpider(scrapy.Spider):
    # Define the name of the spider
    name = "my_spider_img"

    # Specify the starting URL(s) for the spider: the newest page of the Dezeen feed
    start_urls = ["https://www.dezeen.com/feed/?paged=1"]

    # The ImagesPipeline downloads the URLs in "image_urls" into IMAGES_STORE; images smaller than 300 px (icons) are skipped
    custom_settings = {
        "ITEM_PIPELINES": {"scrapy.pipelines.images.ImagesPipeline": 1},
        "IMAGES_STORE": "images",
        "IMAGES_MIN_WIDTH": 300,
        "IMAGES_MIN_HEIGHT": 300,
        "USER_AGENT": "Mozilla/5.0",
        "DOWNLOAD_DELAY": 1,
    }

    def __init__(self, date="2026/09/09", **kwargs):
        super().__init__(**kwargs)
        self.date = date

    def parse(self, response, page=1):
        response.selector.remove_namespaces()
        items = response.xpath("//item")
        for item in items:
            link = item.xpath("link/text()").get()
            if f"/{self.date}/" in link:
                html = item.xpath("encoded/text()").get()
                # the address of every <img> in the article, each one once; the small thumbnails of
                # the "related articles" (their address ends with -191x191.jpg) are left out
                image_urls = scrapy.Selector(text=html).css("img::attr(src)").getall()
                image_urls = [u for u in image_urls if "-191x191" not in u]
                yield {
                    "title": item.xpath("title/text()").get(),
                    "url": link,
                    "image_urls": list(dict.fromkeys(image_urls)),
                }
        if items:
            oldest = re.search(r"dezeen\.com/(\d{4}/\d{2}/\d{2})/", items[-1].xpath("link/text()").get()).group(1)
            if oldest >= self.date:
                yield response.follow(f"https://www.dezeen.com/feed/?paged={page + 1}", callback=self.parse, cb_kwargs={"page": page + 1})

In [ ]:
import json, os, sys
from IPython.display import Image, display

# Run the spider as a separate program. The images are saved in images/full/, and the JSON file lists,
# for every article, the image URLs and the saved file paths (--loglevel=ERROR hides the log).
!"{sys.executable}" -m scrapy runspider my_image_spider.py -a date={date} -O all_scrapy_images_output.json --loglevel=ERROR

with open("all_scrapy_images_output.json", encoding="utf-8") as file:
    results = json.load(file)

for item in results:
    print(item["title"])
    print(f"   {len(item['image_urls'])} image URLs found, {len(item['images'])} images downloaded")

# Show the first three downloaded images
files = sorted(os.listdir("images/full"))
print(f"\n{len(files)} files in images/full")
for name in files[:3]:
    display(Image(filename=os.path.join("images/full", name), width=400))

**Similar to prior tasks, please download the "all_scrapy_images_output.json" from your Google Drive, you need to upload this file to the assignment.**

**Now you have finished the tasks in the notebook, please download the notebook and upload it to the assignment.**